In [1]:
import os 
os.environ["OMP_NUM_THREADS"] = '9'

In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit
from pandas.plotting import autocorrelation_plot
from datetime import datetime

tscv = TimeSeriesSplit() 
print(tscv)
#https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html 

TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None)


In [3]:

usretailcsv = 'us_retail_sales.csv'

retail = pd.read_csv(usretailcsv, header = 0, parse_dates=[0], date_format = '%Y-%m-%d')
retail.columns.tolist()
retail.head(5)

FileNotFoundError: [Errno 2] No such file or directory: 'us_retail_sales.csv'

#### Plot the data with proper labeling and make some observations on the graph. ####

The initial visualization I used was of the raw data to see what the data looked like before data preparation. The first call out is that we did not have full 2021 data available. Most likely when the data was pulled for consumption. Also, immediately I noticed the trend shows every year retail sales increased with a drop around 2009. This may have a correlation with the recession impacting households. The recession ended in June 2009, but economic weakness persisted. Economic growth was only moderate—averaging about 2 percent in the first four years of the recovery—and the unemployment rate, particularly the rate of long-term unemployment, remained at historically elevated levels (Weinberg, 2007 - ). This assumption can be proven in the data as the June 2009 period is where there is a shift in sales. 
Once the data was cleaned and prepared for modeling, a holistic picture is provided. The sales went from $150,00 to over $500,000 from the years 1992 – 2021. The visualization and data was updated to apply the mean to all missing data. In doing so, data visualization and information is ready for the next step of modeling.

References: 
Weingberg, J. (2007 -). The Great Recession and Its Aftermath. Federal Reserve History. https://www.federalreservehistory.org/essays/great-recession-and-its-aftermath 


In [ ]:
retail.plot(x = "YEAR", kind = 'bar', stacked = True)
plt.legend(fontsize = 'x-small') #AI Assist
plt.xticks(rotation = 45, ha = 'right')
plt.xlabel("Year")
plt.ylabel("Month")
plt.title("Annual US Retail Sales by Month")
plt.show()
#https://www.geeksforgeeks.org/python/create-a-stacked-bar-plot-in-matplotlib/

In [ ]:
retail_long = retail.melt(id_vars=['YEAR'], var_name = 'Month', value_name = 'Sales')
retail_long['MONTHS'] = pd.to_datetime(retail_long['Month'], format = '%b').dt.month
retail_long['Date'] = pd.to_datetime(retail_long[['YEAR', 'MONTHS']].assign(day=1))


dt_retail = retail_long.drop(columns =['YEAR', 'Month', 'MONTHS'])
dt_retail = dt_retail.iloc[:,[1,0]]
dt_retail['Sales'] = dt_retail['Sales'].round(2)
#https://www.geeksforgeeks.org/python/change-the-order-of-a-pandas-dataframe-columns-in-python/


# Google Search AI Assist
# dt_retail = dt_retail.set_index('Date')
dt_retail = dt_retail.fillna(dt_retail.mean())
print(dt_retail.head())
dt_retail.dtypes
res  = dt_retail.columns
print(res)

dt_retail.plot(x='Date', kind = 'area')

In [ ]:
from pmdarima import auto_arima
import warnings 
warnings.filterwarnings("ignore")

auto_arima(dt_retail['Sales'], seasonal = True, m=12).summary()

In [ ]:
# autocorrelation_plot(dt_retail);
# dt_retail.plot();
# dt_retail.show()

In [ ]:
# Split Time Series Split
####https://towardsdatascience.com/time-series-analysis-arima-based-models-541de9c7b4db/#### 
# Use the last year of data (July 2020 – June 2021) of data as your test set and the rest as your training set.
# Use the training set to build a predictive model for the monthly retail sales.
from statsmodels.tsa.statespace.sarimax import SARIMAX

df_train = dt_retail[dt_retail['Date'] < '2020-07-01']
df_test = dt_retail[dt_retail['Date'] >= '2020-07-01']

model = SARIMAX(df_train['Sales'], order = (5,1,0), seasonal_order = (1,1,1,12))
result = model.fit()

start = len(df_train)
end = len(df_train) + len(df_test)-1

predictions = result.predict(start = start , end = end , dynamic = False, typ = 'levels').rename('SARIMA(5,1,0)(1,1,1,12) Predictions')

ax = df_test['Sales'].plot(legend=True,figsize=(12,6),title= 'Sales: Actual Vs. Forecasted')
predictions.plot(legend=True)
ax.autoscale(axis='x', tight=True)
ax.set(ylabel='Sales')

In [ ]:
full_model = SARIMAX(dt_retail['Sales'], order=(5,1,0), seasonal_order = (1,1,1,12))

res2 = full_model.fit()

forecast = res2.predict(len(dt_retail),len(dt_retail)+11, typ = 'levels', rename = 'SARIMA Model')

ax = dt_retail['Sales'].plot(legend = True, figsize = (12,6), title = 'Sales: Current to Forecasted')
forecast.plot(legend = True)
ax.autoscale(axis = 'x', tight = True)
ax.set(ylabel = 'Sales')

In [ ]:
#report the RMSE of the model predictions on the test set 

from statsmodels.tools.eval_measures import rmse

errors2 = rmse(df_test['Sales'], predictions)
print(f"The RMSE for the test set is  {errors2}.")